# 5.3 Subgraphs, Supervisor Pattern, Handoffs

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

**Prerequisites:** 4.2 (state, nodes, edges, reducers), 4.4 (interrupts and checkpointers), 5.1 (persistence: threads and checkpointer choice). All demos run with no API key; the supervisor in production would call a model, and the lesson marks exactly where.

## Learning objectives

- Add a compiled subgraph as a node in a parent graph and state the shared-state contract precisely: which keys the child reads and writes, and what breaks when the contract is violated.
- Read `checkpoint_ns` to locate a subgraph's checkpoints, stream subgraph output with `subgraphs=True`, and choose per-invocation versus per-thread persistence for a subgraph.
- Build a supervisor node that routes with `Command(goto=..., update=...)` and explain when a supervisor earns its keep over fixed edges.
- Compare the three handoff styles (direct worker-to-worker, supervisor-mediated, subgraph-to-parent via `Command.PARENT`) and decide between shared state and per-worker state for the transcript.
- Name the failure modes: schema mismatch, `goto` typos, infinite supervisor loops, and misplaced `Command.PARENT` updates.

## Why compose graphs

Lesson 4.2 taught you to build one graph. That works until the graph stops being one thing. A support system with a billing specialist, a technical specialist, and a writer is three different state machines wearing a trench coat: different states, different failure modes, different owners. Splitting them into subgraphs buys four things:

1. **Reuse.** A "research this topic" subgraph can be a node in the support graph, the onboarding graph, and the sales graph without copying code.
2. **Team boundaries.** Two teams can own two subgraphs as long as the input/output keys they agreed on stay stable. The parent never sees the internals.
3. **Private state.** Each agent gets a scratchpad (its own message history, its own intermediate steps) that never pollutes the parent's state. Lesson 5.5's research team leans on this: researchers keep raw source notes to themselves and hand the writer only distilled findings.
4. **Independent persistence.** Each subgraph gets its own checkpoint namespace, so one subagent's memory does not overwrite another's.

The price is indirection. Debugging a three-level graph means reading three levels of state, and a miswired key fails at runtime, not at compile time, as this lesson will show you.

## Subgraph as a node: the shared-state contract

The simplest composition: compile a graph, pass it to `add_node`. The requirement is that parent and child **share state keys**: the child reads and writes the same channels it would if it were a plain node.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict):
    query: str
    findings: Annotated[list, operator.add]

def research(state: ResearchState) -> dict:
    return {"findings": [f"finding-for({state['query']})"]}

child = StateGraph(ResearchState)
child.add_node("research", research)
child.add_edge(START, "research")
child.add_edge("research", END)
research_team = child.compile()          # a finished, compiled graph

class ParentState(TypedDict):
    query: str
    findings: Annotated[list, operator.add]
    report: str

parent = StateGraph(ParentState)
parent.add_node("research_team", research_team)   # the subgraph, as a node
parent.add_node("write", lambda s: {"report": "report-written"})
parent.add_edge(START, "research_team")
parent.add_edge("research_team", "write")
parent.add_edge("write", END)
graph = parent.compile()

out = graph.invoke({"query": "langgraph", "findings": [], "report": ""})
print(out)

Expected output:

In [ ]:
{'query': 'langgraph', 'findings': ['finding-for(langgraph)'], 'report': 'report-written'}

Read what happened mechanically. The parent handed its state to the subgraph node. The subgraph read `query`, appended to `findings` through the shared reducer, and its output merged back into the parent's channels exactly as if `research` had been a plain node. The parent's `report` key, which the child never heard of, passed through untouched.

The contract has two directions and both are strict:

- **Input:** the child receives the parent's state. Every key the child reads must exist in the parent's state.
- **Output:** the child's returned keys merge into the parent's state through the **parent's** reducers. A key the child writes that the parent's schema does not declare is dropped.

There is no compile-time check on this contract. It is a handshake, not a type check.

## What a broken contract looks like

Drop `query` from the parent schema and keep the child as it is:

In [ ]:
class BadParent(TypedDict):
    findings: Annotated[list, operator.add]   # no 'query' key

bad = StateGraph(BadParent)
bad.add_node("research_team", research_team)
bad.add_edge(START, "research_team")
bad.add_edge("research_team", END)
# EXPECTED-FAILURE: raises KeyError: 'query' when the subgraph node runs
bad.compile().invoke({"findings": []})

`compile()` succeeds. The run fails with:

In [ ]:
KeyError: 'query'

The child did `state["query"]` on the parent's state, and the key was not there. This is the failure mode to internalise: subgraph wiring errors are **runtime** errors. They surface when the node executes, possibly deep inside a long run, with a `KeyError` that names the missing key but says nothing about which graph boundary it fell through. When you see a bare `KeyError` inside a subgraph node, check the parent schema first: the key the child needs is missing upstream.

## When the schemas differ: call the subgraph inside a node

Shared keys are not always what you want. A worker agent often needs a private message history that the parent should never see (lesson 5.5's researchers are exactly this case). Then you do not add the subgraph as a node. You invoke it inside a wrapper node and translate both ways:

In [ ]:
class PrivateResearchState(TypedDict):
    messages: Annotated[list, operator.add]

def private_researcher(state: PrivateResearchState) -> dict:
    return {"messages": ["raw note 1", "raw note 2"]}   # scratchpad, stays inside

private_child = StateGraph(PrivateResearchState)
private_child.add_node("researcher", private_researcher)
private_child.add_edge(START, "researcher")
private_child.add_edge("researcher", END)
private_team = private_child.compile()

def research_wrapper(state: ParentState) -> dict:
    # translate parent state -> child input, run, translate child output -> parent update
    result = private_team.invoke({"messages": [f"brief: {state['query']}"]})
    distilled = f"2 findings distilled from {len(result['messages'])} notes"
    return {"findings": [distilled]}

parent2 = StateGraph(ParentState)
parent2.add_node("research_team", research_wrapper)
parent2.add_node("write", lambda s: {"report": "report-written"})
parent2.add_edge(START, "research_team")
parent2.add_edge("research_team", "write")
parent2.add_edge("write", END)
out2 = parent2.compile().invoke({"query": "refunds", "findings": [], "report": ""})
print(out2["findings"])

Expected output:

In [ ]:
['2 findings distilled from 3 notes']

(The child saw the brief plus its two raw notes: three messages total. Only the distilled finding crosses the boundary.)

The wrapper is explicit plumbing, and that is its virtue. The parent's state never contains the raw notes; the child never sees the parent's keys. You pay one function of translation code per boundary. Use the direct `add_node(subgraph)` form when the shared state is genuinely the communication channel (a common `messages` key in a multi-agent team). Use the wrapper form when the child is an implementation detail whose internals the parent must not see.

## `checkpoint_ns`: where a subgraph's checkpoints live

With a checkpointer attached, every checkpoint records which graph it belongs to in `config["configurable"]["checkpoint_ns"]`. The parent's checkpoints carry `""` (the empty string). A subgraph invoked as node `team` carries `"team:<task_id>"`, for example `"team:78c5c600-1025-26d5-d844-9d37bbc52231"`. Nested subgraphs join namespaces with `|`, so a graph inside `team` inside `org` reads `"org:<id>|team:<id>"`.

The namespace matters in three situations:

1. **Debugging.** `graph.get_state(config)` shows the parent. To read the subgraph's state at its latest checkpoint, pass its namespace back in:

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

saver = InMemorySaver()
g = parent.compile(checkpointer=saver)
config = {"configurable": {"thread_id": "demo-1"}}

# one run, collecting the subgraph namespaces from the stream as they appear
namespaces = set()
for ns, chunk in g.stream(
    {"query": "langgraph", "findings": [], "report": ""},
    config,
    stream_mode="updates",
    subgraphs=True,
):
    if ns:
        namespaces.add(ns[0])

snap = g.get_state(config)
print(repr(snap.config["configurable"]["checkpoint_ns"]))   # ''  -> the parent

sub_ns = next(iter(namespaces))
print("subgraph namespace:", sub_ns)
sub = g.get_state({"configurable": {"thread_id": "demo-1", "checkpoint_ns": sub_ns}})
print(sub.values)

Expected output:

In [ ]:
''
subgraph namespace: team:7be79ad1-0f7d-849c-4961-7cfcc9a324f3
{'query': 'langgraph', 'findings': ['finding-for(langgraph)']}

(The task id in the namespace is generated per run; yours will differ. That is exactly why you discover it from the stream rather than typing it by hand.)

The task id in the namespace is generated per run, so you rarely type these strings by hand. For observing a live run, stream instead:

In [ ]:
for ns, chunk in graph.stream(
    {"query": "langgraph", "findings": [], "report": ""},
    stream_mode="updates",
    subgraphs=True,
):
    print(ns, chunk)

Expected output shape (task ids vary per run):

In [ ]:
('team:7be79ad1-...',) {'research': {'findings': ['finding-for(langgraph)']}}
() {'team': {'query': 'langgraph', 'findings': ['finding-for(langgraph)']}}
() {'write': {'report': 'report-written'}}

`subgraphs=True` changes the stream to `(namespace, chunk)` tuples. The empty tuple `()` is the parent; the `('team:...',)` tuple is the subgraph's own event stream. This is the practical way to watch nested execution: filter on the namespace tuple and each level's updates separate themselves.

2. **Persistence mode.** The `checkpointer` argument on the subgraph's `.compile()` controls what its namespace stores, and the three modes from the official docs map to three real needs:

| Mode | `checkpointer=` | Behaviour |
|---|---|---|
| Per-invocation (default) | `None` | Each call starts fresh. Within one call the subgraph inherits the parent's checkpointer, so `interrupt()` still works. Right for one-off subagent tasks. |
| Per-thread | `True` | State accumulates across calls on the same thread: the subagent remembers earlier turns. |
| Stateless | `False` | No checkpoints at all. No interrupts, no durable execution. Runs like a plain function. |

3. **Namespace isolation.** Subgraphs added as nodes get name-based namespaces automatically, so a billing expert and a technical expert never overwrite each other's checkpoints. But if you invoke subgraphs **inside a node function** (the wrapper pattern above), LangGraph assigns namespaces by call order (first call, second call), and reordering the calls mixes up which subgraph loads which state. The fix, per the docs, is to wrap each subagent in its own `StateGraph` with a unique node name so the namespace is stable. A second trap: calling the **same** per-thread subgraph instance twice on one thread writes both calls to the same namespace and they conflict. Per-invocation is the safe default for repeated calls; reach for per-thread only when the subagent genuinely needs cross-call memory.

## The supervisor pattern

A supervisor is one node (often one model call) that looks at the state and decides which worker runs next. Workers do their job and report back to the supervisor, which decides the next step or finishes. The routing vehicle is `Command(goto=..., update=...)`, introduced in lesson 4.2.

Here is the whole pattern with no LLM, so you can run it and read the trace:

In [ ]:
from langgraph.types import Command

class Ticket(TypedDict, total=False):
    request: str
    log: Annotated[list, operator.add]
    handled: bool

def researcher(state: Ticket) -> dict:
    return {"log": [f"researcher: facts for '{state['request']}'"], "handled": True}

def writer(state: Ticket) -> dict:
    return {"log": ["writer: draft written"]}

def supervisor(state: Ticket) -> Command:
    log = state.get("log", [])
    if not any("researcher:" in e for e in log):
        return Command(goto="researcher",
                       update={"log": ["supervisor: assigned research"]})
    if not any("writer:" in e for e in log):
        return Command(goto="writer",
                       update={"log": ["supervisor: assigned writing"]})
    return Command(goto=END, update={"handled": True})

builder = StateGraph(Ticket)
builder.add_node("supervisor", supervisor,
                 destinations=("researcher", "writer", END))
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)
builder.add_edge(START, "supervisor")
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")
supervisor_graph = builder.compile()

out = supervisor_graph.invoke({"request": "reset password", "log": [], "handled": False})
print(out["log"])

Expected output:

In [ ]:
['supervisor: assigned research', "researcher: facts for 'reset password'",
 'supervisor: assigned writing', 'writer: draft written']

Three mechanics to note. First, `destinations=` declares where a `Command`-returning node may route; it is documentation plus validation that the targets exist. Second, the workers have plain edges back to `"supervisor"`, so control always returns to the router after each unit of work. Third, `Command(goto=END, update=...)` ends the run from inside a node, which is how a supervisor finishes without a separate conditional edge.

In production the supervisor node calls a model with the list of workers in its prompt and returns the `Command` for the model's choice. That is the only part that changes; the wiring is identical.

## When a supervisor earns its keep

Fixed edges are cheaper, deterministic, and testable. A supervisor is a model call per routing decision, and every decision can be wrong. Reach for the supervisor when the **sequence of work is not known at build time**: the number of research rounds depends on what the researcher finds, the writer may need to go back for more facts, some tickets need one worker and others need three.

There is a sharper reason, and it comes from a failure practitioners keep reporting: two peer agents that disagree with no one to break the tie. A researcher decides the evidence is thin and loops back for more searching; a writer decides the evidence is enough and wants to proceed. Both are locally correct, and without an authority the system oscillates forever. A recursion limit stops the bleeding but resolves nothing. The supervisor exists to be the tie-breaker: sole authority to review both outputs, decide proceed-or-retry, and lock the decision. Give it a default after N attempts (proceed with the best available output, flagged low-confidence) so deliberation itself is bounded.

The supervisor's costs are real and should be stated plainly. Every routing decision is a model call with its own latency and price. Routing mistakes are silent: the wrong worker runs, produces something plausible, and the supervisor may never notice. And the supervisor's context grows with every worker's output, which is the context-pollution problem the next section addresses. Beyond roughly a handful of workers, the routing prompt gets long enough that routing quality degrades; the documented mitigation is hierarchy, supervisors of supervisors, each with a small team.

## Handoffs: three styles

A handoff is control passing from one agent to another. There are three shapes, and they differ in who decides.

**Direct worker-to-worker.** One worker names the next worker in a `Command`. No supervisor involved:

In [ ]:
class HandoffState(TypedDict, total=False):
    log: Annotated[list, operator.add]
    owner: str

def intake(state: HandoffState) -> Command:
    return Command(goto="billing",
                   update={"log": ["intake: handoff to billing"], "owner": "billing"})

def billing(state: HandoffState) -> dict:
    return {"log": ["billing: resolved"]}

h = StateGraph(HandoffState)
h.add_node("intake", intake, destinations=("billing",))
h.add_node("billing", billing)
h.add_edge(START, "intake")
h.add_edge("billing", END)
out = h.compile().invoke({"log": [], "owner": "intake"})
print(out["log"], "| owner:", out["owner"])

Expected output:

In [ ]:
['intake: handoff to billing', 'billing: resolved'] | owner: billing

Use this when the workflow is a known relay: intake always goes to billing, billing always finishes. The `update` carries the handoff context (here, who owns the ticket now).

**Supervisor-mediated.** The supervisor pattern above is this style: workers never name each other, they return to the supervisor, and only the supervisor routes. Use it when the next step depends on the result, not on the sender.

**Subgraph to parent.** A node inside a subgraph can hand control to a node in the parent graph with `graph=Command.PARENT`. The canonical use is escalation: the specialist team hits something it cannot handle and the parent takes over.

In [ ]:
class Inner(TypedDict, total=False):
    need_parent: bool
    note: str

def inner_node(state: Inner) -> Command:
    if state.get("need_parent"):
        return Command(goto="finalize",
                       update={"note": "escalated"},
                       graph=Command.PARENT)
    return Command(goto=END, update={"note": "handled-inside"})

inner_b = StateGraph(Inner)
inner_b.add_node("inner_node", inner_node)   # no destinations: parent targets are not declared here
inner_b.add_edge(START, "inner_node")
inner_team = inner_b.compile()

class Outer(TypedDict, total=False):
    need_parent: bool
    note: str

outer = StateGraph(Outer)
outer.add_node("inner_team", inner_team)
outer.add_node("finalize", lambda s: {"note": "parent finalized: " + s["note"]})
outer.add_edge(START, "inner_team")
outer.add_edge("inner_team", END)
outer.add_edge("finalize", END)
escalation = outer.compile()

print(escalation.invoke({"need_parent": True, "note": ""})["note"])
print(escalation.invoke({"need_parent": False, "note": ""})["note"])

Expected output:

In [ ]:
parent finalized: escalated
handled-inside

Two subtleties, both verified by running this code. First, the inner node declares **no** `destinations`: the target `finalize` is a parent node, and declaring it inside the subgraph fails compile validation (`Found edge ending at unknown node 'finalize'`). The `graph=Command.PARENT` flag is what carries the command across the boundary. Second, the `update` in a `PARENT` command is applied to the **parent's** state, not the subgraph's. The subgraph never sees `{"note": "escalated"}`; the parent does. If you expected the update to land inside the child, you have the direction backwards.

One honest note on the ecosystem: `langchain.agents.middleware` on the pinned stack (verified by install of langchain 1.4.6) contains **no** handoff middleware. The first-class handoff helpers (`create_handoff_tool`, `transfer_to_<agent>` tools) live in the separate `langgraph-supervisor` package, which this course does not pin or depend on. The `Command` mechanics above are the foundation those helpers are built on; learn them first and the library becomes readable rather than magical.

## Shared state versus per-worker state: who owns the transcript

The deepest design decision in a multi-agent graph is where the conversation lives.

**Shared state** (one `messages` key every worker reads and writes) is simple: every worker sees the full context, handoffs need no translation, and the transcript is automatically complete. The costs are context pollution and price. Every worker's output becomes every other worker's input, so a researcher's ten raw notes sit in the writer's context window, and the supervisor's routing prompt grows with each turn. You pay for those tokens on every subsequent model call, and beyond a point the extra context degrades decisions rather than improving them. There is also a GDPR angle for the EU audience: a shared transcript that accumulates user data across workers is a single growing store of personal data, and lesson 6.5 will ask where it lives and how long it is kept.

**Per-worker state** (each worker or subgraph keeps its own messages; only distilled results merge into shared keys) costs translation code at every boundary but keeps each worker's context small, focused, and cheap. The writer sees findings, not raw notes. The supervisor sees status lines, not transcripts.

The rule of thumb: share the transcript while the team is small (two to three workers) and the task is conversational; split into per-worker state with distilled handoffs as soon as workers produce bulk intermediate output (research notes, tool-call logs, draft revisions) that other workers do not need. Lesson 5.5's research team is built on the split form: researchers own their notes, the writer owns the draft, the shared state carries only briefs, findings, and the final report.

## What can go wrong

- **Schema mismatch across the subgraph boundary.** The parent lacks a key the child reads. `compile()` passes; the run raises `KeyError` inside the child node. Read the traceback's node name, then check the parent's state schema for the missing key. There is no static check, so treat the shared-keys list as an interface and review it like one.
- **`goto` typos are not caught.** A `Command(goto="resercher")` compiles fine and raises nothing at runtime. The framework logs `wrote to unknown channel branch:to:resercher, ignoring it` to stderr, applies the command's `update`, drops the routing, and the run ends as if the goto never happened. The graph finishes "successfully" having skipped the intended node. `destinations=` does not prevent this; it only validates the declared set, not the runtime value. Integration tests that assert the visited node sequence are the real defence.
- **Infinite supervisor loops.** A supervisor whose routing condition never becomes false (the classic form: always route back to the researcher on "needs more") runs until `GraphRecursionError`. The recursion limit is a seatbelt, not a design: add an attempt counter to state, and give the supervisor a default action after N attempts (proceed with best-available, flagged). The disagreement-without-tiebreaker freeze from the motivation section is this bug wearing a trench coat.
- **`Command.PARENT` updates land in the parent.** The update dict in a parent-directed command is applied to the parent's state channels, never the child's. Code that reads the update inside the subgraph after issuing the command is reading stale state.
- **Declaring parent targets in a subgraph's `destinations`.** `add_node(..., destinations=("finalize",))` inside the subgraph fails at compile time because `finalize` is not a node of the subgraph. Parent-directed commands need no `destinations` on the inner node; the `graph=Command.PARENT` flag does the work.
- **Per-thread namespace collisions.** Two calls to the same per-thread subgraph instance on one thread share one checkpoint namespace and overwrite each other. Use per-invocation (the default) for repeated calls; reserve per-thread for subagents that genuinely need cross-call memory, and give each distinct subagent its own node name.
- **The supervisor as serial bottleneck.** Every worker waits for the supervisor's model call before proceeding, and the supervisor re-reads the growing transcript each time. A team of fast workers behind one slow supervisor is slower than a fixed pipeline. If the routing logic can be expressed as rules on state, use a rule-based router node (this lesson's demo) and skip the model call entirely.
- **Checkpointing a transcript you did not mean to keep.** The moment a checkpointer is attached, subgraph namespaces persist every intermediate state, including worker scratchpads and user messages. Decide retention per namespace before production, especially where worker notes contain personal data.

## Check your understanding

**1. A subgraph added with `add_node("team", compiled_child)` reads `state["query"]`, but the parent schema has no `query` key. When does this fail, and with what error?**
It fails at runtime, not at compile time. `compile()` succeeds; when the subgraph node executes, `state["query"]` raises `KeyError: 'query'`. There is no static validation of the shared-keys contract, so a bare `KeyError` inside a subgraph node should send you to the parent's state schema first.

**2. A node returns `Command(goto="resercher", update={"log": ["x"]})`. The graph compiles and the run finishes without an exception, but the researcher never ran. Explain.**
The `goto` target is validated at runtime against the graph's branch channels, and an unknown target is ignored with a stderr warning (`wrote to unknown channel branch:to:resercher, ignoring it`), not an exception. The `update` is still applied (`log` gains `"x"`), the routing is dropped, and the run ends. `destinations=` cannot catch this because it validates the declared set, not the runtime string. The defence is a test asserting the visited node sequence.

**3. In a subgraph-to-parent handoff (`Command(goto="finalize", update={...}, graph=Command.PARENT)`), where is the `update` applied, and why must the inner node declare no `destinations`?**
The update is applied to the parent's state, not the subgraph's; the child never sees it. The inner node must not declare `destinations` because compile validation checks declared targets against the subgraph's own nodes, and the parent node `finalize` is unknown there, so the compile fails with `Found edge ending at unknown node 'finalize'`. The `graph=Command.PARENT` flag carries the command across the boundary on its own.

**4. Two agents disagree forever: the researcher keeps asking for more searching, the writer keeps wanting to proceed, and the run only stops at the recursion limit. What is the design fix, and why is the recursion limit not the fix?**
Give one node, the supervisor, sole authority to break the tie: it reviews both outputs, decides proceed-or-retry, and locks the decision, with a default action after N attempts (proceed with the best available output, flagged low-confidence). The recursion limit only bounds the damage; it resolves nothing and the user gets an error instead of an answer. The bug is missing authority, not missing bounds.

**5. Your supervisor routes well with three workers but starts misrouting with nine. The routing prompt is long. What are the two standard responses, and what does each cost?**
One: split into a hierarchy, supervisors of supervisors, each with a small team, which adds a layer of latency and another component to debug. Two: replace model-based routing with rules where the routing logic is expressible on state (a rule-based router node), which costs flexibility: genuinely ambiguous cases no longer get a model's judgement. Both accept that a single supervisor does not scale linearly with team size.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.